# ChatGPT Conversation History - Categorization & Rating System

This notebook provides advanced categorization, sorting, and rating systems for your conversations.

**Features:**
- Multi-dimensional categorization system
- Comprehensive rating system (Quality, Utility, Depth)
- Advanced sorting and filtering
- Category-based analytics
- Conversation recommendations

---

## Setup

In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from collections import Counter, defaultdict
import re
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("Libraries loaded!")

Libraries loaded!


In [ ]:
# Load data
import os
if not os.path.exists('conversations.json'):
    raise FileNotFoundError("conversations.json not found — put it in the repo root or update the path.")
with open('conversations.json', 'r', encoding='utf-8') as f:
    conversations = json.load(f)

df = pd.read_csv('../chatgpt_conversations_enhanced.csv')
df['create_date'] = pd.to_datetime(df['create_date'])

print(f"Loaded {len(conversations):,} conversations")
print(f"DataFrame shape: {df.shape}")

FileNotFoundError: [Errno 2] No such file or directory: '../chatgpt_backup.json'

## 1. Content-Based Categorization

Categorize conversations based on their content and purpose.

In [ ]:
# Define comprehensive category keywords
CATEGORY_KEYWORDS = {
    'Programming': [
        'code', 'python', 'javascript', 'java', 'function', 'class', 'debug',
        'algorithm', 'programming', 'syntax', 'compile', 'api', 'database',
        'sql', 'react', 'node', 'css', 'html', 'git', 'repository', 'typescript'
    ],
    'Data Science': [
        'data', 'analysis', 'pandas', 'numpy', 'machine learning', 'model',
        'dataset', 'visualization', 'statistics', 'regression', 'neural',
        'tensorflow', 'sklearn', 'matplotlib', 'training', 'prediction', 'ml'
    ],
    'Writing': [
        'write', 'essay', 'article', 'blog', 'story', 'content', 'draft',
        'editing', 'grammar', 'tone', 'creative', 'narrative', 'author',
        'manuscript', 'publication', 'writing'
    ],
    'Education': [
        'learn', 'study', 'homework', 'explain', 'teach', 'lesson', 'course',
        'tutorial', 'understanding', 'concept', 'student', 'education',
        'academic', 'university', 'school', 'exam'
    ],
    'Business': [
        'business', 'marketing', 'strategy', 'sales', 'customer', 'company',
        'market', 'financial', 'revenue', 'product', 'management', 'startup',
        'entrepreneur', 'investment', 'consulting', 'corporate'
    ],
    'Research': [
        'research', 'study', 'paper', 'findings', 'methodology', 'literature',
        'academic', 'journal', 'hypothesis', 'experiment', 'publication',
        'citation', 'thesis', 'scholarly'
    ],
    'Creative': [
        'creative', 'design', 'art', 'music', 'poetry', 'fiction', 'story',
        'character', 'plot', 'worldbuilding', 'inspiration', 'brainstorm',
        'artistic', 'imagination'
    ],
    'Technical Support': [
        'error', 'fix', 'problem', 'issue', 'troubleshoot', 'help', 'solve',
        'broken', 'not working', 'crash', 'bug', 'failing', 'support'
    ],
    'General Knowledge': [
        'what is', 'who is', 'how does', 'why', 'explain', 'tell me', 'fact',
        'information', 'definition', 'history', 'general', 'question'
    ],
    'Career': [
        'career', 'job', 'resume', 'interview', 'hiring', 'professional',
        'workplace', 'salary', 'promotion', 'cv', 'linkedin', 'networking'
    ],
    'Health & Wellness': [
        'health', 'fitness', 'exercise', 'diet', 'nutrition', 'wellness',
        'mental health', 'meditation', 'workout', 'medical'
    ],
    'Planning': [
        'plan', 'schedule', 'organize', 'calendar', 'todo', 'task',
        'project', 'timeline', 'roadmap', 'goals', 'planning'
    ]
}

def categorize_conversation(title, text, threshold=2):
    """Categorize a conversation based on keyword matching."""
    combined = (title + ' ' + text).lower()
    
    category_scores = {}
    for category, keywords in CATEGORY_KEYWORDS.items():
        score = sum(1 for keyword in keywords if keyword in combined)
        if score >= threshold:
            category_scores[category] = score
    
    if category_scores:
        # Return top category
        return max(category_scores.items(), key=lambda x: x[1])[0]
    else:
        return 'Other'

# Extract text for categorization
def extract_text_simple(conv):
    texts = []
    for msg in conv['messages']:
        if msg['role'] in ['user', 'assistant'] and msg['content']:
            texts.append(msg['content'])
    return ' '.join(texts)[:5000]  # Limit to first 5000 chars for efficiency

print("Categorizing conversations...")
categories = []
for i, conv in enumerate(conversations):
    if i % 100 == 0:
        print(f"  Progress: {i}/{len(conversations)}")
    text = extract_text_simple(conv)
    category = categorize_conversation(conv['title'], text)
    categories.append(category)

df['content_category'] = categories
print("✓ Categorization complete!")

In [ ]:
# Category distribution
category_dist = df['content_category'].value_counts()

print("\nCONTENT CATEGORY DISTRIBUTION")
print("=" * 70)
for category, count in category_dist.items():
    pct = (count / len(df)) * 100
    bar = '█' * int(pct / 2)
    print(f"{category:<25} {count:>4} ({pct:>5.1f}%) {bar}")
print("=" * 70)

In [ ]:
# Visualize category distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Bar chart
category_dist_sorted = category_dist.sort_values(ascending=True)
colors = plt.cm.tab20(np.linspace(0, 1, len(category_dist_sorted)))
axes[0].barh(range(len(category_dist_sorted)), category_dist_sorted.values, 
             edgecolor='black', color=colors)
axes[0].set_yticks(range(len(category_dist_sorted)))
axes[0].set_yticklabels(category_dist_sorted.index)
axes[0].set_xlabel('Number of Conversations')
axes[0].set_title('Conversations by Category', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# Pie chart (top categories only)
top_n = 8
top_categories = category_dist.head(top_n)
other_count = category_dist.iloc[top_n:].sum() if len(category_dist) > top_n else 0
if other_count > 0:
    plot_data = pd.concat([top_categories, pd.Series({'Other Categories': other_count})])
else:
    plot_data = top_categories

axes[1].pie(plot_data.values, labels=plot_data.index, autopct='%1.1f%%', 
           startangle=90, colors=plt.cm.tab20(np.linspace(0, 1, len(plot_data))))
axes[1].set_title('Category Distribution (Top Categories)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 2. Length-Based Categorization

Categorize by conversation length and depth.

In [ ]:
# Length categories
def categorize_by_length(message_count):
    if message_count <= 3:
        return 'Quick Q&A'
    elif message_count <= 10:
        return 'Brief Discussion'
    elif message_count <= 20:
        return 'Standard Conversation'
    elif message_count <= 40:
        return 'Extended Discussion'
    else:
        return 'Deep Dive'

df['length_category'] = df['message_count'].apply(categorize_by_length)

length_dist = df['length_category'].value_counts()
print("LENGTH CATEGORY DISTRIBUTION")
print("=" * 70)
for category in ['Quick Q&A', 'Brief Discussion', 'Standard Conversation', 
                 'Extended Discussion', 'Deep Dive']:
    count = length_dist.get(category, 0)
    pct = (count / len(df)) * 100
    bar = '█' * int(pct / 2)
    print(f"{category:<25} {count:>4} ({pct:>5.1f}%) {bar}")

## 3. Interaction Type Categorization

Categorize based on interaction patterns.

In [ ]:
# Interaction type based on ratio and tool usage
def categorize_interaction_type(row):
    ratio = row['ratio']
    tool_msgs = row['tool_messages']
    msg_count = row['message_count']
    
    # High tool usage
    if tool_msgs > 5 and tool_msgs / msg_count > 0.2:
        return 'Tool-Heavy Task'
    
    # Ratio-based categorization
    if ratio > 2.0:
        return 'Detailed Explanation'
    elif ratio < 0.7:
        return 'User-Led Inquiry'
    elif msg_count <= 4:
        return 'Quick Lookup'
    elif 0.8 <= ratio <= 1.5:
        return 'Collaborative Dialog'
    else:
        return 'Information Request'

df['interaction_type'] = df.apply(categorize_interaction_type, axis=1)

interaction_dist = df['interaction_type'].value_counts()
print("\nINTERACTION TYPE DISTRIBUTION")
print("=" * 70)
for itype, count in interaction_dist.items():
    pct = (count / len(df)) * 100
    bar = '█' * int(pct / 2)
    print(f"{itype:<25} {count:>4} ({pct:>5.1f}%) {bar}")

## 4. Comprehensive Rating System

Rate conversations on multiple dimensions.

In [ ]:
# Quality Score (0-100)
# Based on: length, engagement, complexity, topic confidence
def calculate_quality_score(row):
    # Normalize components
    length_score = min(row['message_count'] / 30, 1.0) * 25  # up to 25 points
    engagement_normalized = row['engagement_score'] * 0.30  # up to 30 points
    complexity_score = row['complexity_score'] * 25  # up to 25 points
    confidence_score = row['topic_confidence'] * 20  # up to 20 points
    
    return length_score + engagement_normalized + complexity_score + confidence_score

df['quality_score'] = df.apply(calculate_quality_score, axis=1)

# Utility Score (0-100)
# Based on: tool usage, message count, practical indicators
def calculate_utility_score(row):
    # Tool usage is strong indicator of utility
    tool_score = min(row['tool_messages'] / 10, 1.0) * 40  # up to 40 points
    
    # Length indicates thorough work
    length_score = min(row['message_count'] / 25, 1.0) * 30  # up to 30 points
    
    # Complexity indicates sophisticated usage
    complexity_score = row['complexity_score'] * 30  # up to 30 points
    
    return tool_score + length_score + complexity_score

df['utility_score'] = df.apply(calculate_utility_score, axis=1)

# Depth Score (0-100)
# Based on: message count, ratio balance, engagement
def calculate_depth_score(row):
    # Long conversations = more depth
    length_score = min(row['message_count'] / 40, 1.0) * 40  # up to 40 points
    
    # Balanced ratio = back-and-forth exploration
    ratio_balance = 1 / (1 + abs(row['ratio'] - 1.0))
    balance_score = ratio_balance * 30  # up to 30 points
    
    # High engagement = deep exploration
    engagement_score = (row['engagement_score'] / 100) * 30  # up to 30 points
    
    return length_score + balance_score + engagement_score

df['depth_score'] = df.apply(calculate_depth_score, axis=1)

# Overall Rating (average of all scores)
df['overall_rating'] = (df['quality_score'] + df['utility_score'] + df['depth_score']) / 3

print("✓ Rating system calculated!")
print("\nRATING STATISTICS")
print("=" * 70)
for score_type in ['quality_score', 'utility_score', 'depth_score', 'overall_rating']:
    print(f"\n{score_type.replace('_', ' ').title()}:")
    print(f"  Mean:   {df[score_type].mean():.1f}")
    print(f"  Median: {df[score_type].median():.1f}")
    print(f"  Std:    {df[score_type].std():.1f}")
    print(f"  Min:    {df[score_type].min():.1f}")
    print(f"  Max:    {df[score_type].max():.1f}")

In [ ]:
# Visualize rating distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

scores = ['quality_score', 'utility_score', 'depth_score', 'overall_rating']
titles = ['Quality Score', 'Utility Score', 'Depth Score', 'Overall Rating']
colors_scores = ['skyblue', 'lightcoral', 'lightgreen', 'plum']

for idx, (score, title, color) in enumerate(zip(scores, titles, colors_scores)):
    ax = axes[idx // 2, idx % 2]
    ax.hist(df[score], bins=40, edgecolor='black', alpha=0.7, color=color)
    ax.axvline(df[score].mean(), color='red', linestyle='--', linewidth=2, 
              label=f'Mean: {df[score].mean():.1f}')
    ax.axvline(df[score].median(), color='green', linestyle='--', linewidth=2,
              label=f'Median: {df[score].median():.1f}')
    ax.set_xlabel('Score (0-100)')
    ax.set_ylabel('Frequency')
    ax.set_title(title, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Rating categories
def rate_conversation(score):
    if score >= 80:
        return 'Excellent'
    elif score >= 65:
        return 'Very Good'
    elif score >= 50:
        return 'Good'
    elif score >= 35:
        return 'Fair'
    else:
        return 'Basic'

df['rating_category'] = df['overall_rating'].apply(rate_conversation)

rating_dist = df['rating_category'].value_counts()
print("\nOVERALL RATING DISTRIBUTION")
print("=" * 70)
for rating in ['Excellent', 'Very Good', 'Good', 'Fair', 'Basic']:
    count = rating_dist.get(rating, 0)
    pct = (count / len(df)) * 100
    bar = '★' * int(pct / 5) if pct >= 5 else '☆'
    print(f"{rating:<15} {count:>4} ({pct:>5.1f}%) {bar}")

## 5. Top Rated Conversations

In [ ]:
# Top conversations by overall rating
print("\nTOP 20 HIGHEST RATED CONVERSATIONS")
print("=" * 120)
print(f"{'Rank':<5} {'Rating':<7} {'Quality':<8} {'Utility':<8} {'Depth':<7} {'Category':<20} {'Title':<50}")
print("=" * 120)

top_rated = df.nlargest(20, 'overall_rating')
for rank, (idx, row) in enumerate(top_rated.iterrows(), 1):
    title = row['title'][:47] + '...' if len(row['title']) > 50 else row['title']
    category = row['content_category'][:17] + '...' if len(row['content_category']) > 20 else row['content_category']
    print(f"{rank:<5} {row['overall_rating']:>6.1f} {row['quality_score']:>7.1f} "
          f"{row['utility_score']:>7.1f} {row['depth_score']:>6.1f} {category:<20} {title}")

In [ ]:
# Top by each dimension
print("\n" + "=" * 120)
print("TOP 10 BY QUALITY SCORE")
print("=" * 120)
for rank, (idx, row) in enumerate(df.nlargest(10, 'quality_score').iterrows(), 1):
    title = row['title'][:70] + '...' if len(row['title']) > 70 else row['title']
    print(f"{rank:>2}. [{row['quality_score']:>5.1f}] {title}")

print("\n" + "=" * 120)
print("TOP 10 BY UTILITY SCORE")
print("=" * 120)
for rank, (idx, row) in enumerate(df.nlargest(10, 'utility_score').iterrows(), 1):
    title = row['title'][:70] + '...' if len(row['title']) > 70 else row['title']
    print(f"{rank:>2}. [{row['utility_score']:>5.1f}] {title}")

print("\n" + "=" * 120)
print("TOP 10 BY DEPTH SCORE")
print("=" * 120)
for rank, (idx, row) in enumerate(df.nlargest(10, 'depth_score').iterrows(), 1):
    title = row['title'][:70] + '...' if len(row['title']) > 70 else row['title']
    print(f"{rank:>2}. [{row['depth_score']:>5.1f}] {title}")

## 6. Sorting and Filtering Interface

In [ ]:
def display_conversations(dataframe, sort_by='overall_rating', ascending=False, 
                         limit=20, filters=None):
    """
    Display conversations with sorting and filtering.
    
    Parameters:
    - sort_by: column to sort by
    - ascending: sort order
    - limit: number of results to show
    - filters: dict of column:value pairs to filter by
    """
    result = dataframe.copy()
    
    # Apply filters
    if filters:
        for column, value in filters.items():
            if isinstance(value, list):
                result = result[result[column].isin(value)]
            else:
                result = result[result[column] == value]
    
    # Sort
    result = result.sort_values(sort_by, ascending=ascending)
    
    # Display
    print(f"\nShowing {min(limit, len(result))} of {len(result)} conversations")
    print(f"Sorted by: {sort_by} ({'ascending' if ascending else 'descending'})")
    if filters:
        print(f"Filters: {filters}")
    print("=" * 120)
    
    for i, (idx, row) in enumerate(result.head(limit).iterrows(), 1):
        title = row['title'][:60] + '...' if len(row['title']) > 60 else row['title']
        print(f"{i:>3}. [{row['overall_rating']:>5.1f}] {row['content_category']:<20} | "
              f"{row['length_category']:<25} | {title}")
    
    return result.head(limit)

# Example: Top rated programming conversations
print("\nEXAMPLE: TOP PROGRAMMING CONVERSATIONS")
programming_convs = display_conversations(
    df, 
    sort_by='overall_rating', 
    filters={'content_category': 'Programming'},
    limit=15
)

In [ ]:
# More sorting examples
print("\n" + "="*120)
print("EXAMPLE: MOST RECENT HIGH-QUALITY CONVERSATIONS")
recent_quality = df[df['quality_score'] > 70].sort_values('create_date', ascending=False)
for i, (idx, row) in enumerate(recent_quality.head(10).iterrows(), 1):
    date_str = row['create_date'].strftime('%Y-%m-%d')
    title = row['title'][:60] + '...' if len(row['title']) > 60 else row['title']
    print(f"{i:>2}. [{date_str}] Q:{row['quality_score']:>5.1f} | {row['content_category']:<20} | {title}")

print("\n" + "="*120)
print("EXAMPLE: DEEPEST CONVERSATIONS (EXTENDED DISCUSSIONS)")
deep_convs = display_conversations(
    df,
    sort_by='depth_score',
    filters={'length_category': ['Extended Discussion', 'Deep Dive']},
    limit=10
)

## 7. Category Analytics

In [ ]:
# Average ratings by category
category_stats = df.groupby('content_category').agg({
    'overall_rating': 'mean',
    'quality_score': 'mean',
    'utility_score': 'mean',
    'depth_score': 'mean',
    'message_count': 'mean',
    'title': 'count'
}).round(1)

category_stats.columns = ['Avg Rating', 'Avg Quality', 'Avg Utility', 'Avg Depth', 'Avg Msgs', 'Count']
category_stats = category_stats.sort_values('Avg Rating', ascending=False)

print("\nCATEGORY PERFORMANCE ANALYSIS")
print("=" * 100)
print(f"{'Category':<25} {'Count':<7} {'Rating':<8} {'Quality':<9} {'Utility':<9} {'Depth':<8} {'Avg Msgs'}")
print("=" * 100)
for category, row in category_stats.iterrows():
    cat_display = category[:22] + '...' if len(category) > 25 else category
    print(f"{cat_display:<25} {row['Count']:<7.0f} {row['Avg Rating']:<8.1f} "
          f"{row['Avg Quality']:<9.1f} {row['Avg Utility']:<9.1f} {row['Avg Depth']:<8.1f} {row['Avg Msgs']:.1f}")

In [ ]:
# Visualize category performance
top_categories = category_stats.head(10)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Average rating by category
axes[0, 0].barh(range(len(top_categories)), top_categories['Avg Rating'], 
               edgecolor='black', color='skyblue')
axes[0, 0].set_yticks(range(len(top_categories)))
axes[0, 0].set_yticklabels(top_categories.index)
axes[0, 0].set_xlabel('Average Overall Rating')
axes[0, 0].set_title('Average Rating by Category (Top 10)', fontweight='bold')
axes[0, 0].invert_yaxis()
axes[0, 0].grid(True, alpha=0.3, axis='x')

# Count by category
axes[0, 1].barh(range(len(top_categories)), top_categories['Count'], 
               edgecolor='black', color='lightcoral')
axes[0, 1].set_yticks(range(len(top_categories)))
axes[0, 1].set_yticklabels(top_categories.index)
axes[0, 1].set_xlabel('Number of Conversations')
axes[0, 1].set_title('Conversation Count by Category', fontweight='bold')
axes[0, 1].invert_yaxis()
axes[0, 1].grid(True, alpha=0.3, axis='x')

# Rating components comparison
categories_to_compare = top_categories.head(5).index
x = np.arange(len(categories_to_compare))
width = 0.25

quality_vals = [category_stats.loc[cat, 'Avg Quality'] for cat in categories_to_compare]
utility_vals = [category_stats.loc[cat, 'Avg Utility'] for cat in categories_to_compare]
depth_vals = [category_stats.loc[cat, 'Avg Depth'] for cat in categories_to_compare]

axes[1, 0].bar(x - width, quality_vals, width, label='Quality', edgecolor='black', color='skyblue')
axes[1, 0].bar(x, utility_vals, width, label='Utility', edgecolor='black', color='lightcoral')
axes[1, 0].bar(x + width, depth_vals, width, label='Depth', edgecolor='black', color='lightgreen')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels([cat[:15] for cat in categories_to_compare], rotation=45, ha='right')
axes[1, 0].set_ylabel('Score')
axes[1, 0].set_title('Rating Components by Category (Top 5)', fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Rating distribution across all categories
rating_by_cat = df.groupby('content_category')['overall_rating'].apply(list)
top_cats_for_box = category_stats.head(8).index
box_data = [rating_by_cat[cat] for cat in top_cats_for_box]

bp = axes[1, 1].boxplot(box_data, labels=[cat[:12] for cat in top_cats_for_box], patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
axes[1, 1].set_xticklabels([cat[:12] for cat in top_cats_for_box], rotation=45, ha='right')
axes[1, 1].set_ylabel('Overall Rating')
axes[1, 1].set_title('Rating Distribution by Category', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 8. Cross-Category Analysis

In [ ]:
# Create cross-tabulations
cross_content_length = pd.crosstab(df['content_category'], df['length_category'])
cross_content_interaction = pd.crosstab(df['content_category'], df['interaction_type'])
cross_content_rating = pd.crosstab(df['content_category'], df['rating_category'])

print("\nCONTENT CATEGORY vs LENGTH CATEGORY")
print("=" * 100)
print(cross_content_length.to_string())

print("\n\nCONTENT CATEGORY vs RATING CATEGORY")
print("=" * 100)
print(cross_content_rating.to_string())

In [ ]:
# Heatmap of content vs length
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Normalize for percentage view
cross_content_length_pct = cross_content_length.div(cross_content_length.sum(axis=1), axis=0) * 100

sns.heatmap(cross_content_length_pct, annot=True, fmt='.1f', cmap='YlOrRd', 
           ax=axes[0], cbar_kws={'label': 'Percentage'})
axes[0].set_title('Content Category vs Length Category (% within category)', 
                 fontweight='bold', pad=15)
axes[0].set_xlabel('Length Category')
axes[0].set_ylabel('Content Category')

# Content vs Rating
cross_content_rating_pct = cross_content_rating.div(cross_content_rating.sum(axis=1), axis=0) * 100
sns.heatmap(cross_content_rating_pct, annot=True, fmt='.1f', cmap='RdYlGn',
           ax=axes[1], cbar_kws={'label': 'Percentage'})
axes[1].set_title('Content Category vs Rating Category (% within category)', 
                 fontweight='bold', pad=15)
axes[1].set_xlabel('Rating Category')
axes[1].set_ylabel('Content Category')

plt.tight_layout()
plt.show()

## 9. Conversation Recommendations

Find similar high-quality conversations.

In [ ]:
def recommend_similar_conversations(conversation_idx, top_n=10):
    """
    Recommend similar conversations based on category, rating, and characteristics.
    """
    target = df.loc[conversation_idx]
    
    # Calculate similarity score
    df_copy = df.copy()
    
    # Same category gets bonus
    df_copy['similarity'] = (df_copy['content_category'] == target['content_category']).astype(int) * 30
    
    # Similar length category
    df_copy['similarity'] += (df_copy['length_category'] == target['length_category']).astype(int) * 20
    
    # Similar ratings (normalized difference)
    rating_diff = abs(df_copy['overall_rating'] - target['overall_rating'])
    df_copy['similarity'] += (1 - rating_diff / 100) * 30
    
    # High rated conversations get bonus
    df_copy['similarity'] += (df_copy['overall_rating'] / 100) * 20
    
    # Remove the target conversation
    df_copy = df_copy[df_copy.index != conversation_idx]
    
    # Sort and return
    recommendations = df_copy.nlargest(top_n, 'similarity')
    
    print(f"\nRECOMMENDATIONS SIMILAR TO:")
    print(f"  '{target['title']}'")
    print(f"  Category: {target['content_category']} | Rating: {target['overall_rating']:.1f}")
    print("\n" + "=" * 120)
    
    for i, (idx, row) in enumerate(recommendations.iterrows(), 1):
        title = row['title'][:60] + '...' if len(row['title']) > 60 else row['title']
        print(f"{i:>2}. [Sim: {row['similarity']:.0f}] [Rating: {row['overall_rating']:>5.1f}] "
              f"{row['content_category']:<20} | {title}")
    
    return recommendations

# Example: Find similar to highest rated conversation
top_conv_idx = df['overall_rating'].idxmax()
similar = recommend_similar_conversations(top_conv_idx, top_n=15)

## 10. Save Enhanced Dataset

In [ ]:
# Save comprehensive dataset
df.to_csv('../chatgpt_conversations_categorized_rated.csv', index=False)
print("✓ Saved categorized and rated dataset to 'chatgpt_conversations_categorized_rated.csv'")

# Create summary report
summary = {
    'Total Conversations': len(df),
    'Average Overall Rating': df['overall_rating'].mean(),
    'Top Category': df['content_category'].mode()[0],
    'Most Common Length': df['length_category'].mode()[0],
    'Most Common Interaction': df['interaction_type'].mode()[0],
    'Excellent Rated': (df['rating_category'] == 'Excellent').sum(),
    'Very Good Rated': (df['rating_category'] == 'Very Good').sum(),
    'Average Quality Score': df['quality_score'].mean(),
    'Average Utility Score': df['utility_score'].mean(),
    'Average Depth Score': df['depth_score'].mean()
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv('../chatgpt_analysis_summary.csv', index=False)
print("✓ Saved analysis summary to 'chatgpt_analysis_summary.csv'")

print("\nNew columns added:")
new_cols = ['content_category', 'length_category', 'interaction_type', 
           'quality_score', 'utility_score', 'depth_score', 'overall_rating', 'rating_category']
for col in new_cols:
    print(f"  • {col}")

## 11. Final Insights Summary

In [ ]:
print("\n" + "=" * 100)
print("CATEGORIZATION & RATING SYSTEM - FINAL INSIGHTS")
print("=" * 100)

print("\n📊 CATEGORY BREAKDOWN:")
print(f"  • Total unique content categories: {df['content_category'].nunique()}")
print(f"  • Most common category: {df['content_category'].mode()[0]} "
      f"({(df['content_category'] == df['content_category'].mode()[0]).sum()} conversations)")
print(f"  • Highest rated category: {category_stats.index[0]} "
      f"(avg rating: {category_stats.iloc[0]['Avg Rating']:.1f})")

print("\n⭐ RATING DISTRIBUTION:")
for rating in ['Excellent', 'Very Good', 'Good', 'Fair', 'Basic']:
    count = rating_dist.get(rating, 0)
    pct = (count / len(df)) * 100
    print(f"  • {rating}: {count} conversations ({pct:.1f}%)")

print("\n🎯 QUALITY METRICS:")
print(f"  • Average Overall Rating: {df['overall_rating'].mean():.1f}/100")
print(f"  • Average Quality Score: {df['quality_score'].mean():.1f}/100")
print(f"  • Average Utility Score: {df['utility_score'].mean():.1f}/100")
print(f"  • Average Depth Score: {df['depth_score'].mean():.1f}/100")

print("\n📏 LENGTH PATTERNS:")
for length_cat in ['Quick Q&A', 'Brief Discussion', 'Standard Conversation', 
                   'Extended Discussion', 'Deep Dive']:
    count = length_dist.get(length_cat, 0)
    pct = (count / len(df)) * 100
    avg_rating = df[df['length_category'] == length_cat]['overall_rating'].mean()
    print(f"  • {length_cat}: {count} ({pct:.1f}%) - Avg Rating: {avg_rating:.1f}")

print("\n💬 INTERACTION PATTERNS:")
top_3_interactions = interaction_dist.head(3)
for itype, count in top_3_interactions.items():
    pct = (count / len(df)) * 100
    avg_rating = df[df['interaction_type'] == itype]['overall_rating'].mean()
    print(f"  • {itype}: {count} ({pct:.1f}%) - Avg Rating: {avg_rating:.1f}")

print("\n🏆 TOP PERFORMERS:")
print(f"  • Highest rated conversation: '{df.loc[df['overall_rating'].idxmax(), 'title'][:60]}'")
print(f"    Rating: {df['overall_rating'].max():.1f}")
print(f"  • Most useful conversation: '{df.loc[df['utility_score'].idxmax(), 'title'][:60]}'")
print(f"    Utility: {df['utility_score'].max():.1f}")
print(f"  • Deepest conversation: '{df.loc[df['depth_score'].idxmax(), 'title'][:60]}'")
print(f"    Depth: {df['depth_score'].max():.1f}")

print("\n" + "=" * 100)
print("📁 OUTPUT FILES CREATED:")
print("  • chatgpt_conversations_categorized_rated.csv - Full dataset with all categories and ratings")
print("  • chatgpt_analysis_summary.csv - Summary statistics")
print("\nUse these files for further analysis or to sort/filter conversations!")
print("=" * 100)